# 🖼️ Concevoir et entraîner des CNN — Notebook Bootcamp Data Science

## Module Data Science — Deep Learning

> **Prérequis :** cours `04-DeepLearning_Design_Train_CNN.md`.

Ce notebook rend le cours **exécutable** : on refait à la main les opérations d'un CNN (convolution, padding, stride,
pooling) en NumPy, on vérifie que Keras calcule exactement la même chose, puis on conçoit, entraîne, évalue et améliore
un CNN sur **Fashion-MNIST**, avant de terminer par un **transfer learning** sur de vraies photos.

| Partie | Contenu |
|--------|---------|
| **1** | Une image vue par l'ordinateur |
| **2** | La convolution à la main (NumPy) et la détection de contours |
| **3** | Stride, padding, taille de sortie — et vérification avec Keras |
| **4** | ReLU et pooling |
| **5** | Concevoir l'architecture : dimensions et paramètres |
| **6** | Entraîner le CNN et lire les courbes d'apprentissage |
| **7** | Évaluer : comparaison avec un réseau dense, matrice de confusion, erreurs |
| **8** | Regarder ce que le CNN a appris (filtres, cartes de caractéristiques) |
| **9** | Combattre le surapprentissage : dropout, augmentation, BatchNorm, early stopping |
| **10** | Transfer learning avec MobileNetV2 |
| **11** | Sauvegarder, recharger, prédire |

> ⚡ **Conseil** : sur Google Colab, activez le GPU (*Exécution → Modifier le type d'exécution → GPU*).
> Les parties 6 à 10 s'exécutent alors en quelques minutes. Sur un ordinateur sans GPU, comptez 3 à 4 minutes
> par entraînement de 15 epochs, et davantage pour les parties 9.3 et 10.

In [ ]:
# TensorFlow est déjà installé sur Colab. En local : pip install tensorflow scikit-learn matplotlib pandas
!pip install scikit-learn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

keras.utils.set_random_seed(42)          # reproductibilité (poids initiaux, mélange des données)
np.set_printoptions(precision=3, suppress=True)

print('TensorFlow', tf.__version__, '| GPU disponible :', bool(tf.config.list_physical_devices('GPU')))
print('Environnement prêt ✅')

---
# PARTIE 1 — Une image vue par l'ordinateur

Pour un ordinateur, une image n'est qu'un **tableau de nombres** : hauteur × largeur × canaux.
Fashion-MNIST contient 70 000 images de vêtements en niveaux de gris, de 28 × 28 pixels (valeurs de 0 à 255).

In [ ]:
(X_train_raw, y_train), (X_test_raw, y_test) = keras.datasets.fashion_mnist.load_data()

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print('Train :', X_train_raw.shape, X_train_raw.dtype, '| Test :', X_test_raw.shape)
print('Valeurs des pixels : min', X_train_raw.min(), '| max', X_train_raw.max())
print('Images par classe (train) :', np.bincount(y_train))

In [ ]:
# Un exemple de chaque classe
fig, axes = plt.subplots(1, 10, figsize=(15, 2.2))
for k, ax in enumerate(axes):
    idx = np.where(y_train == k)[0][0]
    ax.imshow(X_train_raw[idx], cmap='gray'); ax.set_title(class_names[k], fontsize=9); ax.axis('off')
plt.show()

In [ ]:
# Zoom sur les valeurs réelles : le coin supérieur gauche d'une bottine
img = X_test_raw[0]
print('Classe :', class_names[y_test[0]])
print(img[8:16, 0:12])        # 8 lignes × 12 colonnes de pixels

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Image 28 × 28')
axes[0].add_patch(plt.Rectangle((-0.5, 7.5), 12, 8, fill=False, color='orange', lw=2))
axes[1].imshow(img[8:16, 0:12], cmap='gray'); axes[1].set_title('Zone encadrée (8 × 12 pixels)')
for ax in axes: ax.axis('off')
plt.show()

> 🔑 Keras attend des images de forme `(hauteur, largeur, canaux)`. Une image en niveaux de gris a **1** canal,
> une image couleur en a **3** (rouge, vert, bleu). Nous ajouterons cet axe en partie 6.

---
# PARTIE 2 — La convolution à la main

On fait glisser un **filtre** $K$ sur l'image ; à chaque position, on multiplie case par case puis on additionne :

$$S(i, j) = \sum_{u}\sum_{v} X(i+u,\; j+v)\, K(u, v) + b$$

In [ ]:
def conv2d(X, K, b=0, stride=1, padding=0):
    # Convolution 2D d'une image X (2D) par un filtre K (2D), avec stride et padding de zéros
    if padding > 0:
        X = np.pad(X, padding)                 # bordure de zéros
    k = K.shape[0]
    h_out = (X.shape[0] - k) // stride + 1
    w_out = (X.shape[1] - k) // stride + 1
    S = np.zeros((h_out, w_out))
    for i in range(h_out):
        for j in range(w_out):
            fenetre = X[i*stride:i*stride+k, j*stride:j*stride+k]
            S[i, j] = np.sum(fenetre * K) + b
    return S

# L'exemple de la Figure 2 du cours
X_ex = np.array([[3, 0, 1, 2, 7],
                 [1, 5, 8, 9, 3],
                 [2, 7, 2, 5, 1],
                 [0, 1, 3, 1, 7],
                 [4, 2, 1, 6, 2]], dtype=float)
K_vert = np.array([[1, 0, -1],
                   [1, 0, -1],
                   [1, 0, -1]], dtype=float)

print('Fenêtre (0,0) × filtre :'); print(X_ex[0:3, 0:3] * K_vert)
print('Somme =', (X_ex[0:3, 0:3] * K_vert).sum())
print('\nCarte de caractéristiques complète :'); print(conv2d(X_ex, K_vert))

## 2.1 Ce que détecte un filtre

Le même filtre appliqué à une image qui contient un **contour vertical** (clair à gauche, sombre à droite) :

In [ ]:
contour = np.array([[10, 10, 10, 0, 0, 0]] * 6, dtype=float)
print('Contour clair → sombre :'); print(conv2d(contour, K_vert))
print('\nContour sombre → clair (image inversée) :'); print(conv2d(contour[:, ::-1], K_vert))
print('\nFiltre de contours HORIZONTAUX sur le même contour vertical :')
print(conv2d(contour, K_vert.T))

> 🧐 Le filtre vertical répond **fort (30)** là où il chevauche le contour, **0** sur les zones uniformes, et le **signe**
> indique le sens du contour. Le filtre horizontal, lui, ne voit rien : chaque filtre est spécialisé dans **un** motif.

## 2.2 Des filtres sur une vraie image

Appliquons quelques filtres classiques à la bottine. Dans un CNN, ces coefficients ne sont pas écrits à la main :
ils sont **appris** (partie 8).

In [ ]:
filtres = {
    'Contours verticaux':   K_vert,
    'Contours horizontaux': K_vert.T,
    'Sobel vertical':       np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]], dtype=float),
    'Flou (moyenne)':       np.ones((3, 3)) / 9,
    'Netteté':              np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=float),
}
img = X_test_raw[0].astype(float) / 255

fig, axes = plt.subplots(1, len(filtres) + 1, figsize=(16, 3))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Image d\'origine')
for ax, (nom, K) in zip(axes[1:], filtres.items()):
    carte = conv2d(img, K, padding=1)
    if 'Contours' in nom or 'Sobel' in nom:              # échelle centrée sur 0 : fond blanc, + en rouge, − en bleu
        m = np.abs(carte).max(); ax.imshow(carte, cmap='RdBu_r', vmin=-m, vmax=m)
    else:
        ax.imshow(carte, cmap='gray')
    ax.set_title(f'{nom}\n{carte.shape}', fontsize=9)
for ax in axes: ax.axis('off')
plt.show()

> 💡 En rouge et en bleu : réponses positives et négatives des détecteurs de contours. Remarquez que le padding de 1
> conserve la taille 28 × 28.

---
# PARTIE 3 — Stride, padding et taille de sortie

$$O = \left\lfloor \frac{N + 2P - K}{S} \right\rfloor + 1$$

In [ ]:
def taille_sortie(N, K, P=0, S=1):
    return (N + 2 * P - K) // S + 1

cas = [(5, 3, 0, 1), (5, 3, 1, 1), (5, 3, 0, 2), (28, 3, 1, 1), (32, 5, 0, 1), (224, 7, 3, 2)]
tableau = pd.DataFrame(cas, columns=['N', 'K', 'P', 'S'])
tableau['O (formule)'] = [taille_sortie(*c) for c in cas]
tableau['O (NumPy)'] = [conv2d(np.zeros((N, N)), np.zeros((K, K)), stride=S, padding=P).shape[0]
                        for N, K, P, S in cas]
tableau

In [ ]:
print('Stride 2 :'); print(conv2d(X_ex, K_vert, stride=2))
print('\nPadding 1 (« same ») :'); print(conv2d(X_ex, K_vert, padding=1))

## 3.1 Keras calcule-t-il la même chose ?

On crée une couche `Conv2D` d'**un seul filtre**, on remplace ses poids par notre filtre `K_vert`, et on compare.

In [ ]:
conv_keras = layers.Conv2D(filters=1, kernel_size=(3, 3), padding='valid', use_bias=False)
x_keras = X_ex.reshape(1, 5, 5, 1)                 # (batch, hauteur, largeur, canaux)
conv_keras.build(x_keras.shape)
conv_keras.set_weights([K_vert.reshape(3, 3, 1, 1)])  # poids Keras : (K, K, canaux_entrée, nb_filtres)

sortie_keras = conv_keras(x_keras).numpy()[0, :, :, 0]
print('Keras :'); print(sortie_keras)
print('\nIdentique à notre NumPy ?', np.allclose(sortie_keras, conv2d(X_ex, K_vert)))

> 🔑 Keras fait **exactement** notre calcul (une corrélation croisée, sans retourner le filtre). La seule différence :
> dans un vrai CNN, les 9 coefficients sont **appris** par backpropagation au lieu d'être fixés.

---
# PARTIE 4 — ReLU et pooling

In [ ]:
def relu(Z):
    return np.maximum(0, Z)

def pool2d(X, taille=2, mode='max'):
    h, w = X.shape[0] // taille, X.shape[1] // taille
    blocs = X[:h * taille, :w * taille].reshape(h, taille, w, taille)
    return blocs.max(axis=(1, 3)) if mode == 'max' else blocs.mean(axis=(1, 3))

carte = conv2d(X_ex, K_vert)
print('Carte :'); print(carte)
print('\nAprès ReLU :'); print(relu(carte))

P_ex = np.array([[1, 3, 2, 1],
                 [4, 6, 5, 0],
                 [3, 1, 1, 2],
                 [2, 8, 7, 4]], dtype=float)
print('\nMax pooling 2×2 (Figure 4) :'); print(pool2d(P_ex))
print('Average pooling 2×2 :');        print(pool2d(P_ex, mode='avg'))

In [ ]:
# La chaîne complète sur la bottine : convolution -> ReLU -> max pooling
carte = conv2d(img, filtres['Sobel vertical'], padding=1)
etapes = [('Image 28×28', img), ('Convolution (Sobel)', carte),
          ('ReLU', relu(carte)), ('Max pooling → 14×14', pool2d(relu(carte)))]

fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for ax, (titre, A) in zip(axes, etapes):
    ax.imshow(A, cmap='gray'); ax.set_title(f'{titre}\n{A.shape}', fontsize=10); ax.axis('off')
plt.show()

> 🧐 ReLU ne garde que les réponses **positives** (un seul sens de contour) ; le max pooling divise la taille par 2
> en conservant les réponses fortes : le contour reste bien visible sur une image 4 fois plus petite.

---
# PARTIE 5 — Concevoir l'architecture : dimensions et paramètres

$$\text{paramètres d'une convolution} = (K \times K \times C_{\text{entrée}} + 1) \times C_{\text{sortie}}$$

On calcule à la main le tableau de la section 10.2 du cours, puis on le compare à `model.summary()`.

In [ ]:
def params_conv(K, c_in, c_out):  return (K * K * c_in + 1) * c_out
def params_dense(n_in, n_out):    return n_in * n_out + n_out

a_la_main = pd.DataFrame([
    ('Conv 3×3, 32 filtres', '28×28×32', params_conv(3, 1, 32)),
    ('MaxPool 2×2',          '14×14×32', 0),
    ('Conv 3×3, 64 filtres', '14×14×64', params_conv(3, 32, 64)),
    ('MaxPool 2×2',          '7×7×64',   0),
    ('Flatten',              '3136',     0),
    ('Dense 128',            '128',      params_dense(7 * 7 * 64, 128)),
    ('Dense 10',             '10',       params_dense(128, 10)),
], columns=['couche', 'sortie', 'paramètres'])
print(a_la_main.to_string(index=False))
print('\nTotal à la main :', a_la_main['paramètres'].sum())

In [ ]:
def creer_cnn(dropout=True):
    return keras.Sequential([
        keras.Input(shape=(28, 28, 1)),
        # --- Extraction de caractéristiques ---
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),
        # --- Classification ---
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5 if dropout else 0.0),
        layers.Dense(10, activation='softmax'),
    ])

model = creer_cnn()
model.summary()
print('Total Keras = total à la main ?', model.count_params() == a_la_main['paramètres'].sum())

> ⚠️ La couche `Dense(128)` concentre 95 % des paramètres. Gardez-le en tête : c'est elle qui surapprend (partie 9).

---
# PARTIE 6 — Entraîner le CNN

## 6.1 Préparer les données

In [ ]:
X_train = X_train_raw.astype('float32')[..., np.newaxis] / 255.0   # (60000, 28, 28, 1), valeurs dans [0, 1]
X_test  = X_test_raw.astype('float32')[..., np.newaxis] / 255.0

print('X_train :', X_train.shape, '| min', X_train.min(), '| max', X_train.max())
print('y_train :', y_train[:10], '(étiquettes entières → sparse_categorical_crossentropy)')

## 6.2 Compiler et entraîner

`validation_split=0.1` réserve 6 000 images d'entraînement pour **surveiller** le modèle ; le jeu de test reste
intact jusqu'à l'évaluation finale.

In [ ]:
keras.utils.set_random_seed(42)
model = creer_cnn()
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(X_train, y_train, epochs=15, batch_size=64,
                    validation_split=0.1, verbose=2)

In [ ]:
def tracer_courbes(histories, titre=''):
    # Trace loss et accuracy (train en trait plein, validation en pointillés) pour un ou plusieurs modèles
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for nom, h in histories.items():
        h = h.history if hasattr(h, 'history') else h
        ep = np.arange(1, len(h['loss']) + 1)
        l, = axes[0].plot(ep, h['loss'], lw=2, label=f'{nom} — train')
        axes[0].plot(ep, h['val_loss'], '--', lw=2, color=l.get_color(), label=f'{nom} — validation')
        axes[1].plot(ep, h['accuracy'], lw=2, color=l.get_color(), label=f'{nom} — train')
        axes[1].plot(ep, h['val_accuracy'], '--', lw=2, color=l.get_color(), label=f'{nom} — validation')
    for ax, m in zip(axes, ['Loss', 'Accuracy']):
        ax.set_xlabel('Epoch'); ax.set_ylabel(m); ax.grid(alpha=.3); ax.legend(fontsize=8)
    fig.suptitle(titre); plt.tight_layout(); plt.show()

tracer_courbes({'CNN': history}, 'Courbes d\'apprentissage du CNN')
h = history.history
best = int(np.argmin(h['val_loss'])) + 1
print(f"Meilleure loss de validation : {min(h['val_loss']):.4f} à l'epoch {best}")
print(f"Epoch finale : accuracy train {h['accuracy'][-1]:.4f} | validation {h['val_accuracy'][-1]:.4f}")

> 🧐 **Lecture** : tant que les deux courbes descendent ensemble, le modèle apprend des règles générales.
> Quand la loss de validation cesse de baisser alors que celle d'entraînement continue, il commence à **mémoriser**.

---
# PARTIE 7 — Évaluer

## 7.1 Sur le jeu de test, et face à un réseau dense

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'CNN — accuracy test : {test_acc:.4f}')

# Référence : un réseau dense entraîné dans les mêmes conditions
keras.utils.set_random_seed(42)
mlp = keras.Sequential([keras.Input(shape=(28, 28, 1)), layers.Flatten(),
                        layers.Dense(128, activation='relu'), layers.Dense(10, activation='softmax')])
mlp.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
mlp.fit(X_train, y_train, epochs=15, batch_size=64, validation_split=0.1, verbose=0)
mlp_loss, mlp_acc = mlp.evaluate(X_test, y_test, verbose=0)

pd.DataFrame({
    'modèle': ['Réseau dense (MLP)', 'CNN'],
    'paramètres': [mlp.count_params(), model.count_params()],
    'accuracy test': [mlp_acc, test_acc],
    'erreurs / 10 000': [round((1 - mlp_acc) * 10000), round((1 - test_acc) * 10000)],
}).round(4)

> 🧐 Le CNN commet nettement moins d'erreurs (dans le cours : 824 contre 1 249). Ce n'est pas son nombre de paramètres
> qui fait la différence — 95 % d'entre eux sont dans la couche dense — mais les **convolutions**, qui exploitent la
> structure locale de l'image.

## 7.2 Matrice de confusion et rapport par classe

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_proba = model.predict(X_test, verbose=0)
y_pred = y_proba.argmax(axis=1)
print(classification_report(y_test, y_pred, target_names=class_names, digits=3))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(10), class_names, rotation=45, ha='right'); ax.set_yticks(range(10), class_names)
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=8,
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
ax.set_xlabel('Classe prédite'); ax.set_ylabel('Classe réelle'); ax.set_title('Matrice de confusion — CNN')
plt.tight_layout(); plt.show()

In [ ]:
# Les confusions les plus fréquentes
cm_off = cm.copy(); np.fill_diagonal(cm_off, 0)
paires = sorted(((cm_off[i, j], class_names[i], class_names[j]) for i in range(10) for j in range(10)), reverse=True)[:5]
pd.DataFrame(paires, columns=['nombre', 'classe réelle', 'prédite comme'])

In [ ]:
# Quelques erreurs : l'image, la vraie classe, la prédiction et sa probabilité
erreurs = np.where(y_pred != y_test)[0][:12]
fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for ax, i in zip(axes.ravel(), erreurs):
    ax.imshow(X_test_raw[i], cmap='gray'); ax.axis('off')
    ax.set_title(f'vrai : {class_names[y_test[i]]}\nprédit : {class_names[y_pred[i]]} ({y_proba[i].max():.0%})',
                 fontsize=9, color='firebrick')
plt.tight_layout(); plt.show()

> 🧐 Les erreurs portent sur des vêtements qui se ressemblent vraiment en 28 × 28 pixels (chemise / t-shirt / pull / manteau),
> et certaines « erreurs » sont même discutables à l'œil nu. Ce n'est pas du hasard : c'est la limite de l'information
> disponible dans l'image.

---
# PARTIE 8 — Regarder ce que le CNN a appris

## 8.1 Les filtres de la première couche

In [ ]:
W1, b1 = model.layers[0].get_weights()         # W1 : (3, 3, 1, 32)
print('Forme des poids de la 1re convolution :', W1.shape, '| biais :', b1.shape)

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for k, ax in enumerate(axes.ravel()):
    ax.imshow(W1[:, :, 0, k], cmap='RdBu_r', vmin=-np.abs(W1).max(), vmax=np.abs(W1).max())
    ax.set_title(f'filtre {k}', fontsize=8); ax.axis('off')
fig.suptitle('Les 32 filtres 3×3 appris (rouge : poids positifs, bleu : négatifs)')
plt.tight_layout(); plt.show()

> 💡 Plusieurs filtres ressemblent à des **détecteurs de contours** (une moitié rouge, une moitié bleue), comme ceux
> de la partie 2 — mais personne ne les a programmés : le réseau les a découverts pour réduire sa loss.

## 8.2 Les cartes de caractéristiques

In [ ]:
extracteur = keras.Model(inputs=model.inputs,
                         outputs=[model.layers[0].output, model.layers[2].output])
cartes1, cartes2 = extracteur.predict(X_test[:1], verbose=0)
print('Conv 1 :', cartes1.shape, '| Conv 2 :', cartes2.shape)

for cartes, titre in [(cartes1, 'Première convolution (28×28)'), (cartes2, 'Deuxième convolution (14×14)')]:
    plus_actives = np.argsort(cartes[0].mean(axis=(0, 1)))[::-1][:8]
    fig, axes = plt.subplots(1, 9, figsize=(16, 2.2))
    axes[0].imshow(X_test_raw[0], cmap='gray'); axes[0].set_title('image', fontsize=9)
    for ax, k in zip(axes[1:], plus_actives):
        ax.imshow(cartes[0, :, :, k], cmap='viridis'); ax.set_title(f'filtre {k}', fontsize=9)
    for ax in axes: ax.axis('off')
    fig.suptitle(titre, y=1.08); plt.show()

> 🧐 Les cartes de la première couche ressemblent encore à la bottine (contours, semelle, zones claires).
> Celles de la deuxième sont plus petites et plus abstraites : elles signalent la **présence** de motifs.

---
# PARTIE 9 — Combattre le surapprentissage

## 9.1 Le rôle du dropout

Même CNN, **sans** dropout, entraîné dans les mêmes conditions.

In [ ]:
keras.utils.set_random_seed(42)
model_sans = creer_cnn(dropout=False)
model_sans.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_sans = model_sans.fit(X_train, y_train, epochs=15, batch_size=64, validation_split=0.1, verbose=0)

tracer_courbes({'sans dropout': history_sans, 'avec dropout': history}, 'Effet du dropout')

resume = []
for nom, h, m in [('sans dropout', history_sans.history, model_sans), ('avec dropout', history.history, model)]:
    resume.append({'modèle': nom,
                   'loss val. min (epoch)': f"{min(h['val_loss']):.3f} ({int(np.argmin(h['val_loss'])) + 1})",
                   'loss val. finale': round(h['val_loss'][-1], 3),
                   'acc. train finale': round(h['accuracy'][-1], 4),
                   'acc. val. finale': round(h['val_accuracy'][-1], 4),
                   'acc. test': round(m.evaluate(X_test, y_test, verbose=0)[1], 4)})
pd.DataFrame(resume)

> 🧐 Sans dropout, la loss d'entraînement plonge mais la loss de validation **remonte** après quelques epochs :
> le modèle mémorise. Avec dropout, l'écart train / validation reste faible et le modèle généralise mieux.

## 9.2 L'augmentation de données

Les couches d'augmentation transforment aléatoirement chaque image **à chaque epoch** (et uniquement à l'entraînement).

In [ ]:
augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomTranslation(0.1, 0.1),
], name='augmentation')

exemple = X_train[:1]
fig, axes = plt.subplots(1, 8, figsize=(14, 2.2))
axes[0].imshow(exemple[0, :, :, 0], cmap='gray'); axes[0].set_title('originale', fontsize=9)
for ax in axes[1:]:
    ax.imshow(augmentation(exemple, training=True)[0, :, :, 0], cmap='gray'); ax.set_title('augmentée', fontsize=9)
for ax in axes: ax.axis('off')
plt.show()

# En inférence (training=False), l'image n'est pas modifiée
print('Inchangée en inférence ?', np.allclose(augmentation(exemple, training=False), exemple))

> ⚠️ Un miroir horizontal convient aux vêtements, **pas** aux chiffres ni aux lettres : une transformation doit
> toujours préserver la classe.

## 9.3 Tout combiner : BatchNorm, augmentation, Global Average Pooling, early stopping

> ⏱️ C'est l'entraînement le plus long du notebook (chaque epoch est plus coûteuse à cause de l'augmentation et des
> quatre convolutions). Sur GPU : quelques minutes. Sur CPU : jusqu'à une trentaine de minutes.

In [ ]:
def bloc_conv(filtres):
    return [layers.Conv2D(filtres, (3, 3), padding='same', use_bias=False),
            layers.BatchNormalization(),
            layers.Activation('relu')]

keras.utils.set_random_seed(42)
model2 = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    augmentation,
    *bloc_conv(32), *bloc_conv(32), layers.MaxPooling2D(), layers.Dropout(0.25),
    *bloc_conv(64), *bloc_conv(64), layers.MaxPooling2D(), layers.Dropout(0.25),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax'),
], name='cnn_ameliore')
model2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print('Paramètres :', model2.count_params(), 'contre', model.count_params(), 'pour le premier CNN')

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
]
history2 = model2.fit(X_train, y_train, epochs=40, batch_size=64,
                      validation_split=0.1, callbacks=callbacks, verbose=2)

In [ ]:
h2 = history2.history
print(f"Epochs effectuées : {len(h2['loss'])} | meilleure epoch (val_loss) : {int(np.argmin(h2['val_loss'])) + 1}")
tracer_courbes({'CNN amélioré': history2}, 'CNN amélioré (avec early stopping)')

acc2 = model2.evaluate(X_test, y_test, verbose=0)[1]
ecart = h2['accuracy'][-1] - h2['val_accuracy'][-1]
print(f"Accuracy train {h2['accuracy'][-1]:.4f} | validation {h2['val_accuracy'][-1]:.4f} | écart {ecart:+.4f}")
print('Diagnostic :', 'SOUS-apprentissage (train ≤ validation) : trop de régularisation' if ecart <= 0
      else 'surapprentissage modéré' if ecart > 0.02 else 'bon ajustement')
pd.DataFrame({
    'modèle': ['Réseau dense', 'CNN (section 6)', 'CNN amélioré'],
    'paramètres': [mlp.count_params(), model.count_params(), model2.count_params()],
    'accuracy test': [mlp_acc, test_acc, acc2],
}).round(4)

> 🧐 **Comparez honnêtement les trois lignes.** Le Global Average Pooling divise le nombre de paramètres par plus de 5,
> et `EarlyStopping(restore_best_weights=True)` choisit seul la meilleure epoch. Mais dans le cours, ce modèle « amélioré »
> obtient **90,65 %**, soit **moins** que le CNN simple (91,76 %) : son accuracy d'entraînement reste sous celle de
> validation, il **sous-apprend**. Trop de régularisation d'un coup (dropout partout, peu de filtres, décalages
> d'images alors que les vêtements de Fashion-MNIST sont toujours centrés).
>
> 🔑 **La leçon** : on régularise **pour corriger un surapprentissage observé**, à petites doses, en vérifiant l'effet
> sur la validation. Essayez : 64 puis 128 filtres, `RandomTranslation(0.05, 0.05)`, moins de dropout.

---
# PARTIE 10 — Transfer learning avec MobileNetV2

On réutilise **MobileNetV2**, pré-entraîné sur ImageNet (1,2 million d'images), pour classer de vraies photos de
**fleurs** (5 classes, 3 670 photos couleur). Le principe est identique pour des photos de feuilles malades, de fruits
ou de radiographies : il suffit d'un dossier par classe.

> 📥 Le jeu de données (≈ 220 Mo) et les poids de MobileNetV2 (≈ 9 Mo) sont téléchargés automatiquement.

In [ ]:
import pathlib

archive = keras.utils.get_file(
    'flower_photos',
    origin='https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz',
    extract=True)
racine = next(p.parent for p in pathlib.Path(archive).rglob('daisy') if p.is_dir())   # dossier contenant les 5 classes
print('Dossier :', racine)
print({d.name: len(list(d.glob('*.jpg'))) for d in sorted(racine.iterdir()) if d.is_dir()})

In [ ]:
IMG_SIZE = (160, 160)
train_ds, val_ds = keras.utils.image_dataset_from_directory(
    racine, image_size=IMG_SIZE, batch_size=32,
    validation_split=0.2, subset='both', seed=42)
noms_fleurs = train_ds.class_names
print('Classes :', noms_fleurs)

images, etiquettes = next(iter(train_ds))
fig, axes = plt.subplots(1, 6, figsize=(15, 2.8))
for ax, im, e in zip(axes, images, etiquettes):
    ax.imshow(im.numpy().astype('uint8')); ax.set_title(noms_fleurs[e]); ax.axis('off')
plt.show()

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

## 10.1 Phase 1 — Extraction de caractéristiques (base gelée)

In [ ]:
base = keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base.trainable = False                                      # 🔒 on gèle la base pré-entraînée

augmentation_photos = keras.Sequential([layers.RandomFlip('horizontal'),
                                        layers.RandomRotation(0.05),
                                        layers.RandomZoom(0.1)], name='augmentation_photos')

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = augmentation_photos(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)    # SON prétraitement : pixels -> [-1, 1]
x = base(x, training=False)                                 # BatchNorm de la base en mode inférence
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(noms_fleurs), activation='softmax')(x)
model_tl = keras.Model(inputs, outputs)

entrainables = sum(int(np.prod(w.shape)) for w in model_tl.trainable_weights)
print(f'Couches de la base : {len(base.layers)} | paramètres totaux : {model_tl.count_params():,}'
      f' | entraînables : {entrainables:,}')

model_tl.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_tl = model_tl.fit(train_ds, validation_data=val_ds, epochs=5, verbose=2)

> 🧐 Seuls **6 405** paramètres (la nouvelle tête) sont entraînés sur 2,26 millions. Lors de notre exécution,
> l'accuracy de validation atteignait déjà **82 %** après une seule epoch et **86 %** après cinq, avec moins de
> 3 000 photos d'entraînement : les caractéristiques apprises sur ImageNet (contours, textures, formes) sont réutilisables.

## 10.2 Phase 2 — Fine-tuning (dernières couches dégelées, learning rate très petit)

In [ ]:
base.trainable = True
for couche in base.layers[:-30]:                            # on garde gelées toutes les couches sauf les 30 dernières
    couche.trainable = False

model_tl.compile(optimizer=keras.optimizers.Adam(1e-5),     # recompiler après tout changement de trainable
                 loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_ft = model_tl.fit(train_ds, validation_data=val_ds, epochs=5, verbose=2)

val_acc = history_tl.history['val_accuracy'] + history_ft.history['val_accuracy']
plt.figure(figsize=(8, 3.5))
plt.plot(range(1, len(val_acc) + 1), val_acc, 'o-', lw=2)
plt.axvline(len(history_tl.history['val_accuracy']) + 0.5, color='gray', ls='--', label='début du fine-tuning')
plt.xlabel('Epoch'); plt.ylabel('Accuracy de validation'); plt.grid(alpha=.3); plt.legend(); plt.show()

> 🧐 Lors de notre exécution, le fine-tuning a fait passer l'accuracy de validation de **86,2 %** à **88,6 %** en cinq epochs :
> ajuster légèrement les dernières couches de la base les adapte aux fleurs, sans détruire ce qu'elles ont appris sur ImageNet.

> ⚠️ Les quatre règles : utiliser le **prétraitement du modèle** (`preprocess_input`), **geler** la base en phase 1,
> appeler la base avec **`training=False`**, et **recompiler** avec un learning rate **très petit** pour le fine-tuning.

---
# PARTIE 11 — Sauvegarder, recharger, prédire

In [ ]:
model.save('cnn_fashion.keras')                             # architecture + poids + état de l'optimiseur
modele_recharge = keras.models.load_model('cnn_fashion.keras')

identiques = np.allclose(modele_recharge.predict(X_test[:100], verbose=0),
                         model.predict(X_test[:100], verbose=0))
print('Prédictions identiques après rechargement ?', identiques)

# Prédire une image : ajouter l'axe du batch -> (1, 28, 28, 1)
i = 0
probas = modele_recharge.predict(X_test[i][np.newaxis], verbose=0)[0]
top3 = np.argsort(probas)[::-1][:3]
print('\nVraie classe :', class_names[y_test[i]])
for k in top3:
    print(f'  {class_names[k]:<12} {probas[k]:.3f}')

> ⚠️ Pour une **nouvelle** image (une photo prise au téléphone), appliquez exactement le même prétraitement :
> niveaux de gris, 28 × 28, division par 255, et inversion des couleurs (`1 - x`) car les vêtements de Fashion-MNIST
> sont **clairs sur fond noir**. Une image trop différente des données d'entraînement sera souvent mal classée.

---
# ✅ Synthèse

| Étape | Ce que vous avez vu |
|-------|---------------------|
| **Convolution** | un filtre glisse sur l'image ; la carte obtenue indique **où** son motif est présent ; Keras calcule exactement la même chose que notre NumPy |
| **Stride, padding** | $O = \lfloor (N + 2P - K)/S \rfloor + 1$ ; `same` conserve la taille |
| **ReLU, pooling** | garder les réponses positives, puis résumer (÷ 2) sans aucun paramètre |
| **Architecture** | paramètres $(K \cdot K \cdot C_{in} + 1) \cdot C_{out}$ ; la couche dense après Flatten domine le total |
| **Entraînement** | ÷ 255, axe canal, `compile(adam, sparse_categorical_crossentropy)`, `fit(validation_split)` |
| **Évaluation** | le CNN bat nettement le réseau dense ; les erreurs portent sur des classes proches |
| **Ce que voit le CNN** | des filtres de contours appris seuls, des cartes de plus en plus abstraites |
| **Surapprentissage** | dropout, augmentation, BatchNorm, Global Average Pooling, early stopping |
| **Transfer learning** | base pré-entraînée gelée + nouvelle tête, puis fine-tuning à lr = 1e-5 |

### 🚀 Pour aller plus loin
- Ajoutez un troisième bloc convolutif (128 filtres) au CNN de la partie 5 : recalculez les paramètres à la main, puis comparez l'accuracy test.
- Entraînez le CNN sur seulement 5 000 images (`X_train[:5000]`), avec puis sans augmentation : mesurez l'écart train / validation.
- Remplacez `Flatten` par `GlobalAveragePooling2D` dans `creer_cnn` : combien de paramètres gagnez-vous, et que devient l'accuracy ?
- Refaites la partie 10 avec vos propres photos (un sous-dossier par classe), par exemple des feuilles saines et malades.

---
*📘 Module Data Science — Concevoir et entraîner des CNN | Bootcamp Data Science*